# Critical-input DEQN: postprocess trained policies

Run this after `00`, `01`, `02`, `04`, and `05` have finished.  It loads the saved checkpoints and writes simulation artifacts for figures, ergodic distributions, and policy comparisons.

In [ ]:
# Configure simulation and deterministic IRF artifact settings.
from pathlib import Path
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT = ARTIFACT_ROOT / 'postprocess'
OUT.mkdir(parents=True, exist_ok=True)

LENGTH = 2_000
BATCH_SIZE = 64
SEED = 777
IR_BURNIN = 400
IR_HORIZON = 200
IR_PRESTEPS = 5
IR_RELIEF_LAG = 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
print(ROOT)
print(OUT)

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)



In [ ]:
# Generate simulated paths and controlled shock-scenario artifacts.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.postprocess',
    '--artifact-root', str(ARTIFACT_ROOT),
    '--output-dir', str(OUT),
    '--length', str(LENGTH),
    '--batch-size', str(BATCH_SIZE),
    '--seed', str(SEED),
    '--ir-burnin', str(IR_BURNIN),
    '--ir-horizon', str(IR_HORIZON),
    '--ir-presteps', str(IR_PRESTEPS),
    '--ir-relief-lag', str(IR_RELIEF_LAG),
    '--device', DEVICE,
    '--dtype', DTYPE,
]
run_stream(cmd, cwd=ROOT)

In [ ]:
# List saved postprocess artifacts for tables and figures.
sorted(p.name for p in OUT.glob('*'))